In [1]:
#============================================================
# AI-Driven Student Performance Prediction System
# Linear Regression + Logistic Regression
# Author: Archit Vishnoi
# ============================================================
 
# ============================================================
# 1. Import Required Libraries
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)


# ============================================================
# 2. Load Dataset
# ============================================================

df = pd.read_csv("../dataset/new.csv")
print("Dataset Loaded Successfully!")
print(f"Dataset Shape: {df.shape}")
 
# ============================================================
# 3. Data Preprocessing
# ============================================================
# Target variable for Linear Regression
y_reg = df["Final_Exam_Score"]
 
# Remove unnecessary columns
X = df.drop(
    columns=[
        "Student_ID",
        "Semester_ID",
        "University_Name",
        "Home_City",
        "Final_Exam_Score",
    ]
)
 
# Identify categorical columns
categorical_columns = X.select_dtypes(include=["object", "string"]).columns
print("\nCategorical Columns:")
print(categorical_columns)
 
# Convert categorical columns into numerical format
X = pd.get_dummies(X, columns=categorical_columns, drop_first=True)
print("\nFeature Matrix Shape:", X.shape)
 
# ============================================================
# 4. Train-Test Split for Linear Regression
# ============================================================
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
 
# ============================================================
# 5. Linear Regression Model
# ============================================================
linear_model = LinearRegression()
linear_model.fit(X_train_reg, y_train_reg)
y_pred_reg = linear_model.predict(X_test_reg)
 
# ============================================================
# 6. Linear Regression Evaluation
# ============================================================
mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
 
print("\n========== Linear Regression ==========")
prediction_df = pd.DataFrame({"Actual": y_test_reg, "Predicted": y_pred_reg})
print(prediction_df.head(10))
print("\nMean Squared Error:", mse)
print("R² Score:", r2)
 
# ============================================================
# 7. Create Performance Categories
# ============================================================
def categorize(score):
    if score < 50:
        return "Poor"
    elif score < 70:
        return "Average"
    elif score < 85:
        return "Good"
    else:
        return "Excellent"
 
 
df["Performance"] = df["Final_Exam_Score"].apply(categorize)
print("\nPerformance Categories:")
print(df[["Final_Exam_Score", "Performance"]].head(10))
 
# ============================================================
# 8. Prepare Target for Logistic Regression
# ============================================================
y_class = df["Performance"]
label_encoder = LabelEncoder()
y_class = label_encoder.fit_transform(y_class)
print("\nClass Labels:")
print(label_encoder.classes_)
 
# ============================================================
# 9. Train-Test Split for Logistic Regression
# ============================================================
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_class, test_size=0.2, random_state=42
)
 
# ============================================================
# 10. Logistic Regression Model
# ============================================================
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_cls, y_train_cls)
y_pred_cls = logistic_model.predict(X_test_cls)
 
# ============================================================
# 11. Logistic Regression Evaluation
# ============================================================
accuracy = accuracy_score(y_test_cls, y_pred_cls)
print("\n========== Logistic Regression ==========")
print("Accuracy:", accuracy)
print("\nConfusion Matrix")
print(confusion_matrix(y_test_cls, y_pred_cls))
print("\nClassification Report")
print(classification_report(y_test_cls, y_pred_cls))
 
# ============================================================
# 12. Convert Predictions Back to Labels (Optional)
# ============================================================
predicted_labels = label_encoder.inverse_transform(y_pred_cls)
actual_labels = label_encoder.inverse_transform(y_test_cls)
 
results = pd.DataFrame({"Actual": actual_labels, "Predicted": predicted_labels})
print("\nSample Predictions:")
print(results.head(10))
 
# ------------------------------------------------------------
# PCA (first pass)
# ------------------------------------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

# Display PCA results
print("Original Shape:", X.shape)
print("Reduced Shape:", X_pca.shape)
print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)
print("Total Variance Retained:", sum(pca.explained_variance_ratio_))
 
# ============================================================
# 13. Linear Regression with PCA
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
 
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
    X_pca, y_reg, test_size=0.2, random_state=42
)
 
linear_model_pca = LinearRegression()
linear_model_pca.fit(X_train_pca, y_train_pca)
y_pred_pca = linear_model_pca.predict(X_test_pca)
 
mse_pca = mean_squared_error(y_test_pca, y_pred_pca)
r2_pca = r2_score(y_test_pca, y_pred_pca)
 
print("\n========== Linear Regression with PCA ==========")
print("Mean Squared Error:", mse_pca)
print("R² Score:", r2_pca)
 
comparison = pd.DataFrame(
    {
        "Model": ["Linear Regression", "Linear Regression + PCA"],
        "MSE": [mse, mse_pca],
        "R² Score": [r2, r2_pca],
    }
)
print("\n========== Model Comparison ==========")
print(comparison)
 
# ============================================================
# 14. Save Models
# ============================================================

# ============================================================
# 14. Save Models
# ============================================================

import os
import joblib

# Save models in the PROJECT ROOT (not notebooks/models)
MODEL_DIR = "../models"

os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(linear_model, os.path.join(MODEL_DIR, "linear_model.pkl"))
joblib.dump(logistic_model, os.path.join(MODEL_DIR, "logistic_model.pkl"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.pkl"))
joblib.dump(pca, os.path.join(MODEL_DIR, "pca.pkl"))
joblib.dump(label_encoder, os.path.join(MODEL_DIR, "label_encoder.pkl"))
joblib.dump(X.columns.tolist(), os.path.join(MODEL_DIR, "feature_columns.pkl"))

print("✅ Models saved successfully!")

# Verify
for file in os.listdir(MODEL_DIR):
    size = os.path.getsize(os.path.join(MODEL_DIR, file))
    print(file, "-", size, "bytes")

print(df["Gender"].unique())

print(df["Region_Type"].unique())

print(df["Major_Subject"].unique())

print(df["Parent_Education"].unique())

print(df["Family_Income_Level"].unique())

print(df.columns.tolist())

Dataset Loaded Successfully!
Dataset Shape: (500000, 27)

Categorical Columns:
Index(['Gender', 'Region_Type', 'Major_Subject', 'Parent_Education',
       'Family_Income_Level'],
      dtype='str')

Feature Matrix Shape: (500000, 46)

========== Linear Regression ==========
        Actual  Predicted
104241    44.2  50.886819
199676    64.7  61.539794
140199    52.9  45.710984
132814    50.6  54.635579
408697    58.2  60.098716
163280    60.9  58.299398
215758    50.5  54.561737
442316    40.0  45.173852
6940      55.0  55.964508
382310    49.8  46.305620

Mean Squared Error: 56.56024887057388
R² Score: 0.557291641667353

Performance Categories:
   Final_Exam_Score Performance
0               5.6        Poor
1              45.7        Poor
2              51.2     Average
3              41.0        Poor
4              65.8     Average
5              77.5        Good
6              41.2        Poor
7              44.1        Poor
8              53.2     Average
9              62.0     Ave

c:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



========== Logistic Regression ==========
Accuracy: 0.74143

Confusion Matrix
[[57838    25  1664  4312]
 [   86   486   328     3]
 [ 8251    19  3325     0]
 [11066     1   102 12494]]

Classification Report
              precision    recall  f1-score   support

           0       0.75      0.91      0.82     63839
           1       0.92      0.54      0.68       903
           2       0.61      0.29      0.39     11595
           3       0.74      0.53      0.62     23663

    accuracy                           0.74    100000
   macro avg       0.76      0.56      0.63    100000
weighted avg       0.73      0.74      0.72    100000


Sample Predictions:
    Actual Predicted
0     Poor   Average
1  Average   Average
2  Average      Poor
3  Average   Average
4  Average   Average
5  Average   Average
6  Average   Average
7     Poor   Average
8  Average   Average
9     Poor      Poor
Original Shape: (500000, 46)
Reduced Shape: (500000, 39)
Explained Variance Ratio:
[0.05071954 0.03694